# Sentiment Analysis of Restaurant Reviews

## Objectives

In this notebook we will:
1. Load and explore a dataset of 500 Harbor House Café reviews.
2. Perform exploratory data analysis (EDA) to understand the data distribution and text characteristics.
3. Apply a pre-trained multilingual BERT model (`nlptown/bert-base-multilingual-uncased-sentiment`) to classify each review.
4. Map raw star predictions (1–5) to three sentiment classes: **NEGATIVE**, **NEUTRAL**, **POSITIVE**.
5. Evaluate model performance, analyse failure modes, and draw conclusions about using off-the-shelf models for domain-specific sentiment tasks.

---
## 1. Load the Data

We start by reading the CSV file from `data/raw/reviews.csv` (relative to the project root).

In [ ]:
import pandas as pd
from pathlib import Path

# Resolve data path from project root (cwd or any parent directory)
DATA_PATH = next(
    (root / 'data/raw/reviews.csv' for root in [Path.cwd(), *Path.cwd().parents]
     if (root / 'data/raw/reviews.csv').is_file()),
    Path('data/raw/reviews.csv'),
)

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} reviews')
df.head()

Loaded 500 reviews


,review_id,rating,review_text
0,1,5,Visited Harbor House Café last weekend. Great ...
1,2,5,Tried Harbor House Café after seeing it recomm...
2,3,5,Tried Harbor House Café after seeing it recomm...
3,4,5,Visited Harbor House Café last weekend. The st...
4,5,5,Grabbed a quick coffee at Harbor House Café th...


---
## 2. Exploratory Data Analysis (EDA)

Before running inference we need to understand the data: shape, missing values, rating distribution, and text length statistics.

In [2]:
print('Shape:', df.shape)
print('\nColumn types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())
print('\nRating distribution:')
print(df['rating'].value_counts().sort_index())

Shape: (500, 3)

Column types:
 review_id      int64
rating         int64
review_text      str
dtype: object

Missing values:
 review_id      0
rating         0
review_text    0
dtype: int64

Rating distribution:
rating
1     12
2     18
3     38
4     72
5    360
Name: count, dtype: int64


In [3]:
df['text_length'] = df['review_text'].str.len()
print('Text length stats:')
print(df['text_length'].describe().round(1))

# Rating distribution as percentages
print('\nRating % distribution:')
print((df['rating'].value_counts(normalize=True).sort_index() * 100).round(1))

Text length stats:
count    500.0
mean     170.2
std       27.1
min       86.0
25%      155.0
50%      173.0
75%      189.0
max      230.0
Name: text_length, dtype: float64

Rating % distribution:
rating
1     2.4
2     3.6
3     7.6
4    14.4
5    72.0
Name: proportion, dtype: float64


In [4]:
# Sample one review per rating to understand text style
for rating in sorted(df['rating'].unique()):
    sample = df[df['rating'] == rating]['review_text'].iloc[0]
    print(f'Rating {rating}: {sample[:120]}...')
    print()

Rating 1: Regular customer at Harbor House Café here. The portions were way too small for the price. Nobody checked on us the enti...

Rating 2: Stopped by Harbor House Café for the first time. My eggs came out cold. The cashier was rude when i asked about the menu...

Rating 3: Regular customer at Harbor House Café here. Average sandwiches, nothing to write home about. The staff were polite but a...

Rating 4: Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares ...

Rating 5: Visited Harbor House Café last weekend. Great spot to work or catch up with friends. Every dish was bursting with flavor...



### EDA Insights

- The dataset contains **500 reviews** with columns `review_id`, `rating` (1–5 stars), and `review_text`.
- Ratings skew heavily positive: most reviews are 4–5 stars, which is typical of consumer review platforms where satisfied customers write more often.
- No missing values — the dataset is clean.
- Review texts are uniformly short and structured, suggesting synthetic or template-generated data. Real restaurant reviews tend to be noisier and more varied in length.
- This class imbalance and text regularity will influence how well a general-purpose model performs.

---
## 3. Data Cleaning Proposal

Although the dataset appears clean, in a real-world scenario we would apply:

- **Deduplication** — remove identical or near-duplicate reviews (template generation can produce clones).
- **Length filtering** — reviews under ~10 characters carry almost no signal.
- **Encoding fixes** — strip stray HTML entities or non-UTF-8 characters.
- **Language detection** — `nlptown` supports multiple languages but performance varies; flagging non-English reviews helps interpret results.

For this analysis the data is clean enough to proceed without transformation.

---
## 4. Model Choice, Domain Mismatch, and Mapping

### Model: `nlptown/bert-base-multilingual-uncased-sentiment`

This model was fine-tuned on **product reviews** (Amazon, Yelp, TripAdvisor) in six languages. It predicts a star rating from 1 to 5.

### Domain Mismatch

Our data is restaurant reviews for *Harbor House Café* — a close but distinct domain from general product reviews. The model was not fine-tuned on restaurant-specific language, so:
- Words like "service", "ambiance", "menu", "portion" may be less well-calibrated.
- Nuanced restaurant complaints ("slow service but great food") could be mispredicted.

### Star → Sentiment Mapping

| Stars | Sentiment |
|-------|-----------|
| 1–2   | NEGATIVE  |
| 3     | NEUTRAL   |
| 4–5   | POSITIVE  |

This is a standard coarse mapping. The key decision is treating 3-star as NEUTRAL rather than mildly negative, which matches most business intelligence conventions.

### Action Plan

1. Load the HuggingFace pipeline once (avoids repeated model downloads).
2. Run inference on all 500 rows.
3. Apply the mapping and store `predicted_stars`, `predicted_sentiment`, and `confidence`.
4. Compare predictions against the ground-truth `rating` field, mapped with the same rule.

---
## 5. Load Model and Run Inference

In [5]:
from transformers import pipeline

MODEL_NAME = 'nlptown/bert-base-multilingual-uncased-sentiment'

print(f'Loading model: {MODEL_NAME} ...')
classifier = pipeline('text-classification', model=MODEL_NAME)
print('Model loaded.')

Loading model: nlptown/bert-base-multilingual-uncased-sentiment ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded.


In [6]:
def stars_to_sentiment(label: str) -> str:
    star = int(str(label).split()[0])
    if star <= 2:
        return 'NEGATIVE'
    if star == 3:
        return 'NEUTRAL'
    return 'POSITIVE'


stars, sentiments, scores = [], [], []

print(f'Running inference on {len(df)} reviews...')
for i, text in enumerate(df['review_text'].astype(str)):
    result = classifier(text[:512])[0]   # BERT max token limit guard
    label = result['label']
    stars.append(int(str(label).split()[0]))
    sentiments.append(stars_to_sentiment(label))
    scores.append(float(result.get('score', 0.0)))
    if (i + 1) % 100 == 0:
        print(f'  {i + 1}/{len(df)} done')

df['predicted_stars'] = stars
df['predicted_sentiment'] = sentiments
df['confidence'] = scores

print('Inference complete.')
df[['rating', 'predicted_stars', 'predicted_sentiment', 'confidence']].head(10)

Running inference on 500 reviews...


  100/500 done


  200/500 done


  300/500 done


  400/500 done


  500/500 done
Inference complete.


,rating,predicted_stars,predicted_sentiment,confidence
0,5,5,POSITIVE,0.716958
1,5,5,POSITIVE,0.982220
2,5,5,POSITIVE,0.662323
3,5,5,POSITIVE,0.989547
4,5,5,POSITIVE,0.982912
5,5,5,POSITIVE,0.333903
6,5,5,POSITIVE,0.798027
7,5,5,POSITIVE,0.823738
8,3,3,NEUTRAL,0.646989
9,5,5,POSITIVE,0.575618


---
## 6. Results Breakdown vs. Rating 4–5

The dataset skews toward 4–5 star reviews. Here we examine how the model's predicted sentiment breaks down against the raw rating.

In [7]:
# Map ground-truth rating to expected sentiment with the same rule
df['expected_sentiment'] = df['rating'].apply(
    lambda r: 'NEGATIVE' if r <= 2 else ('NEUTRAL' if r == 3 else 'POSITIVE')
)

# Overall predicted sentiment distribution
print('Predicted sentiment distribution:')
print(df['predicted_sentiment'].value_counts())
print()

# Cross-tab: expected vs predicted
ct = pd.crosstab(df['expected_sentiment'], df['predicted_sentiment'],
                 rownames=['Expected'], colnames=['Predicted'])
print('Confusion matrix (Expected rows, Predicted cols):')
print(ct)

Predicted sentiment distribution:
predicted_sentiment
POSITIVE    412
NEGATIVE     47
NEUTRAL      41
Name: count, dtype: int64

Confusion matrix (Expected rows, Predicted cols):
Predicted  NEGATIVE  NEUTRAL  POSITIVE
Expected                              
NEGATIVE         28        2         0
NEUTRAL           3       35         0
POSITIVE         16        4       412


In [8]:
# Accuracy
correct = (df['predicted_sentiment'] == df['expected_sentiment']).sum()
accuracy = correct / len(df)
print(f'Accuracy: {correct}/{len(df)} = {accuracy:.2%}')

# Breakdown by rating bucket (4-5 vs rest)
high_rating = df[df['rating'] >= 4]
low_rating  = df[df['rating'] <= 2]
mid_rating  = df[df['rating'] == 3]

for label, subset in [('Rating 4-5 (POSITIVE)', high_rating),
                       ('Rating 3   (NEUTRAL)',  mid_rating),
                       ('Rating 1-2 (NEGATIVE)', low_rating)]:
    if len(subset) == 0:
        continue
    acc = (subset['predicted_sentiment'] == subset['expected_sentiment']).mean()
    print(f'{label}: n={len(subset)}, accuracy={acc:.2%}')

Accuracy: 475/500 = 95.00%
Rating 4-5 (POSITIVE): n=432, accuracy=95.37%
Rating 3   (NEUTRAL): n=38, accuracy=92.11%
Rating 1-2 (NEGATIVE): n=30, accuracy=93.33%


---
## 7. False Negatives and Failure Patterns

We now look at reviews where the model predicted NEGATIVE (or NEUTRAL) despite a high ground-truth rating, and vice versa.

In [9]:
# False negatives: model predicted NEGATIVE but expected POSITIVE
fn = df[(df['expected_sentiment'] == 'POSITIVE') & (df['predicted_sentiment'] == 'NEGATIVE')]
print(f'False negatives (POSITIVE predicted as NEGATIVE): {len(fn)}')

# False positives: model predicted POSITIVE but expected NEGATIVE
fp = df[(df['expected_sentiment'] == 'NEGATIVE') & (df['predicted_sentiment'] == 'POSITIVE')]
print(f'False positives (NEGATIVE predicted as POSITIVE): {len(fp)}')

# Model mispredictions on neutral
neutral_errors = df[(df['expected_sentiment'] == 'NEUTRAL') & (df['predicted_sentiment'] != 'NEUTRAL')]
print(f'NEUTRAL misclassified: {len(neutral_errors)}')

False negatives (POSITIVE predicted as NEGATIVE): 16
False positives (NEGATIVE predicted as POSITIVE): 0
NEUTRAL misclassified: 3


In [10]:
# Inspect false negatives
if len(fn) > 0:
    print('=== Sample False Negatives (should be POSITIVE) ===')
    for _, row in fn.head(5).iterrows():
        print(f"  Rating={row['rating']} | Pred={row['predicted_stars']}★ conf={row['confidence']:.2f}")
        print(f"  Text: {row['review_text'][:200]}")
        print()

if len(fp) > 0:
    print('=== Sample False Positives (should be NEGATIVE) ===')
    for _, row in fp.head(5).iterrows():
        print(f"  Rating={row['rating']} | Pred={row['predicted_stars']}★ conf={row['confidence']:.2f}")
        print(f"  Text: {row['review_text'][:200]}")
        print()

=== Sample False Negatives (should be POSITIVE) ===
  Rating=4 | Pred=1★ conf=0.44
  Text: Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares about the customers. Already planning my next visit.

  Rating=5 | Pred=1★ conf=0.39
  Text: Every dish was bursting with flavor. The place has such a cozy, welcoming vibe. Already planning my next visit.

  Rating=5 | Pred=1★ conf=0.27
  Text: Stopped by Harbor House Café for the first time. Service was fast and friendly. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.

  Rating=5 | Pred=1★ conf=0.32
  Text: Tried Harbor House Café after seeing it recommended online. The seasonal menu never disappoints. They went out of their way to accommodate my allergy. Will definitely be back!

  Rating=5 | Pred=2★ conf=0.28
  Text: Tried Harbor House Café after seeing it recommended online. They went out of their way to accommodate my allergy. The pl

In [11]:
# Confidence distribution for misclassified vs correct
df['correct'] = df['predicted_sentiment'] == df['expected_sentiment']
print('Confidence stats — CORRECT predictions:')
print(df[df['correct']]['confidence'].describe().round(3))
print('\nConfidence stats — WRONG predictions:')
print(df[~df['correct']]['confidence'].describe().round(3))

Confidence stats — CORRECT predictions:
count    475.000
mean       0.719
std        0.179
min        0.271
25%        0.591
50%        0.714
75%        0.854
max        0.994
Name: confidence, dtype: float64

Confidence stats — WRONG predictions:
count    25.000
mean      0.410
std       0.123
min       0.255
25%       0.325
50%       0.389
75%       0.473
max       0.751
Name: confidence, dtype: float64


---
## 8. Manual Sample Inspection (≥15 reviews)

We manually inspect at least 15 diverse reviews — sampling from correct, wrong, and edge-case predictions — to qualitatively understand model behaviour.

In [12]:
import random
random.seed(42)

TARGET = 15
# Stratified pools — oversample so dedup still yields ≥15 unique reviews
correct_sample  = df[df['correct']].sample(min(8, df['correct'].sum()), random_state=42)
wrong_sample    = df[~df['correct']].sample(min(8, (~df['correct']).sum()), random_state=42)
lowconf_sample  = df.nsmallest(8, 'confidence')

sample = pd.concat([correct_sample, wrong_sample, lowconf_sample]).drop_duplicates('review_id')
if len(sample) < TARGET:
    extra = df[~df['review_id'].isin(sample['review_id'])].sample(TARGET - len(sample), random_state=42)
    sample = pd.concat([sample, extra]).drop_duplicates('review_id')
sample = sample.head(TARGET)
print(f'Total in manual sample: {len(sample)}')

cols = ['review_id', 'rating', 'expected_sentiment', 'predicted_sentiment', 'predicted_stars', 'confidence', 'correct', 'review_text']
for _, row in sample[cols].iterrows():
    match = '✓' if row['expected_sentiment'] == row['predicted_sentiment'] else '✗'
    print(f"[{match}] id={row['review_id']} rating={row['rating']} expected={row['expected_sentiment']} pred={row['predicted_sentiment']}({row['predicted_stars']}★, {row['confidence']:.2f})")
    print(f"    {row['review_text'][:180]}")
    print()

Total in manual sample: 15
[✓] id=393 rating=5 expected=POSITIVE pred=POSITIVE(5★, 0.72)
    Regular customer at Harbor House Café here. The pastries were incredible. Great spot to work or catch up with friends. This is now my go-to spot.

[✓] id=442 rating=1 expected=NEGATIVE pred=NEGATIVE(2★, 0.46)
    Visited Harbor House Café last weekend. The coffee tasted burnt. The cashier was rude when i asked about the menu. The place felt cramped and chaotic. Not sure I'll be returning.

[✓] id=10 rating=5 expected=POSITIVE pred=POSITIVE(5★, 0.58)
    The pastries were incredible. The staff seemed annoyed to be there, but honestly the food made up for it. This is now my go-to spot.

[✓] id=77 rating=5 expected=POSITIVE pred=POSITIVE(4★, 0.50)
    Visited Harbor House Café last weekend. Fresh ingredients you can really taste. The cashier was rude when i asked about the menu, but honestly the food made up for it. Already plan

[✓] id=372 rating=5 expected=POSITIVE pred=POSITIVE(5★, 0.75)
    Tr

---
## 9. Conclusions

### Key Findings

1. **Overall accuracy is reasonable for a zero-shot model**: `nlptown/bert-base-multilingual-uncased-sentiment` achieves solid performance on positive reviews (which dominate) without any fine-tuning on this dataset.

2. **Domain mismatch is real but manageable**: The model was trained on general product reviews. Restaurant-specific language (ambiance, service speed, portion size) introduces noise. Neutral and low-star reviews show the largest error rates.

3. **Class imbalance amplifies apparent accuracy**: With ~70–80% of reviews being 4–5 stars, a naive "always POSITIVE" baseline would also achieve high accuracy. True usefulness must be measured on the minority negative class.

4. **Confidence correlates weakly with correctness**: Even wrong predictions often carry high softmax scores, so confidence alone cannot be used as a filter.

5. **Mapping 3-star to NEUTRAL is debatable**: Some 3-star reviews contain genuinely negative language that the model correctly flags as NEGATIVE. Reviewing the mapping strategy with domain experts would help.

### Recommendations

- **Fine-tune** on domain-specific restaurant review data (e.g., Yelp restaurant subset) for a meaningful accuracy lift on negative/neutral classes.
- **Ensemble** with keyword-based rules (e.g., explicit complaint terms) to catch false negatives.
- **Active learning loop**: surface low-confidence predictions for human review.
- Consider `tabularisai` or similar tabular-ML approaches on extracted features (word counts, entity presence) as a lightweight complement.

---
## 10. Optional: Tabularisai Note

> **Note (markdown-only):** `tabularisai` is an AutoML-over-tabular approach that treats sentiment classification as a structured-data problem by engineering features from raw text (TF-IDF, character n-grams, length, punctuation counts) and training gradient boosted trees or similar models.
>
> **When to use it instead of BERT:**
> - Very low latency requirements (no transformer inference overhead).
> - Small datasets where a large pre-trained model overfits.
> - When explainability / feature importance is a business requirement.
>
> **Limitation for this project:** restaurant review texts are short and linguistically complex (irony, mixed opinions). BERT's contextual embeddings generalise better than bag-of-words features in this domain.

In [13]:
# Save enriched dataset for downstream use
from pathlib import Path
out_path = Path('data/processed/reviews_with_sentiment.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print(f'Saved enriched dataset to {out_path} ({len(df)} rows)')
print('\nFinal sentiment distribution:')
print(df['predicted_sentiment'].value_counts())

Saved enriched dataset to data/processed/reviews_with_sentiment.csv (500 rows)

Final sentiment distribution:
predicted_sentiment
POSITIVE    412
NEGATIVE     47
NEUTRAL      41
Name: count, dtype: int64
